In [2]:
import re
from pathlib import Path

import numpy as np
import imageio.v2 as imageio 
import matplotlib.pyplot as plt

In [8]:

# =================== Konfiguration ===================

FPS           = 10
NX, NY        = 576, 576             

CLIP_MIN      = 0.0
CLIP_MAX      = 1.0

TYPE_COLORS = {
    1: (0.0, 1.0, 1.0),  # cyan
    2: (0.0, 1.0, 0.0),  # grün
    3: (1.0, 0.0, 0.0),  # rot
    4: (1.0, 1.0, 0.0),  # gelb
    5: (0.0, 1.0, 0.0),  # grün
    6: (0.0, 1.0, 1.0),  # cyan
}

################################
# type funcs
################################
def get_cell_c(cell_id):
    
    
    p = cell_id // 2 #which pair 
    part = cell_id % 2   #re/im

    if p % 3 == 0:
        return TYPE_COLORS[2] if part == 0 else TYPE_COLORS[5]
    elif p % 3 == 1:
        return TYPE_COLORS[3] if part == 0 else TYPE_COLORS[4]
    else:
        return TYPE_COLORS[1] if part == 0 else TYPE_COLORS[6]

def get_cell_c_TS_L(cell_id):
    
    if cell_id >= 108:
        return (1.0, 0.0, 1.0)
    
    p = cell_id // 2 #which pair 
    part = cell_id % 2   #re/im
    
    if p % 3 == 0:
        return TYPE_COLORS[5] if part == 0 else TYPE_COLORS[2]
    elif p % 3 == 1:
        return TYPE_COLORS[4] if part == 0 else TYPE_COLORS[3]
    else:
        return TYPE_COLORS[6] if part == 0 else TYPE_COLORS[1]
    
def get_cell_c_TS(cell_id):


    p = cell_id // 2 #which pair 
    part = cell_id % 2   #re/im
    
    if p % 3 == 0:
        return TYPE_COLORS[5] if part == 0 else TYPE_COLORS[2]
    elif p % 3 == 1:
        return TYPE_COLORS[4] if part == 0 else TYPE_COLORS[3]
    else:
        return TYPE_COLORS[6] if part == 0 else TYPE_COLORS[1]
   
    
################################
# colouring funcs
################################
def fields_to_rgb(fields, colours, vmin=CLIP_MIN, vmax=CLIP_MAX):

    rgb = np.zeros((NY, NX, 3), dtype=np.float32)
    if vmax - vmin <= 1e-12:
        scale = 1.0
        offset = vmin
    else:
        scale = 1.0 / (vmax - vmin)
        offset = vmin
    for field,colour in zip(fields,colours):
        field_c = np.clip(field, vmin, vmax)
        f = (field_c - offset) * scale   
        f = np.clip(f, 0.0, 1.0)
        rgb[..., 0] += f * colour[0]
        rgb[..., 1] += f * colour[1]
        rgb[..., 2] += f * colour[2]
    rgb = np.clip(rgb, 0.0, 1.0)
    frame = (rgb * 255).astype(np.uint8)
    return frame


def fields_to_viridis(fields, vmin=CLIP_MIN, vmax=CLIP_MAX):

    rgb = np.zeros((NY, NX, 3), dtype=np.float32)
    cmap = plt.get_cmap("viridis")

    if vmax - vmin <= 1e-12:
        scale = 1.0
        offset = vmin
    else:
        scale = 1.0 / (vmax - vmin)
        offset = vmin

    for field in fields:
        field_c = np.clip(field, vmin, vmax)
        f = (field_c - offset) * scale
        f = np.clip(f, 0.0, 1.0)

        colour_field = cmap(f)[..., :3]

        rgb[..., 0] += f * colour_field[..., 0]
        rgb[..., 1] += f * colour_field[..., 1]
        rgb[..., 2] += f * colour_field[..., 2]

    rgb = np.clip(rgb, 0.0, 1.0)
    frame = (rgb * 255).astype(np.uint8)
    return frame



def fields_to_rgb_line(fields, colours):

    rgb = np.zeros((NY, NX, 3), dtype=np.float32)

    for field, colour in zip(fields, colours):

        #border visible w. error
        f = (np.abs(field - 0.5) < 0.0777).astype(np.float32)

        rgb[..., 0] += f * colour[0]
        rgb[..., 1] += f * colour[1]
        rgb[..., 2] += f * colour[2]

    rgb = np.clip(rgb, 0.0, 1.0)

    frame = (rgb * 255).astype(np.uint8)

    return frame

def fields_to_rgb_area(fields, colours):

    rgb = np.zeros((NY, NX, 3), dtype=np.float32)

    for field, colour in zip(fields, colours):

        #border visible w. error
        # f = ((field - 0.5) > 0).astype(np.float32)
        field_max = np.max(field)
        if field_max >=0.5:
            f = (field >= field_max - 0.5).astype(np.float32)
        else:
            f=0

        rgb[..., 0] += f * colour[0]
        rgb[..., 1] += f * colour[1]
        rgb[..., 2] += f * colour[2]

    rgb = np.clip(rgb, 0.0, 1.0)

    frame = (rgb * 255).astype(np.uint8)

    return frame



In [13]:
FRAMES_DIR    = Path(r"E:\MASTER THESIS\code\out_debug_ver2b_lcell_3_6_5_5_5_0.5")    #unpacked folder of sim run (not .tar.gz!!!)

PATTERN_RHO = r"^rho(\d+)-t([0-9.eE+-]+)\.txt$"
# PATTERN_RHO = r"^(rho2?)-t([0-9.eE+-]+)\.txt$"

OUT_VIDEO     = "sub/ver2b_lcell_3_6_5_5_5_0.5.mp4"  #
NX, NY        = 576, 576     
times = {}
found_cells = set()

################################
# get filename for every cell in every time
# 
################################
for f in FRAMES_DIR.iterdir():
    m = re.search(PATTERN_RHO, f.name)
    if not m:
        continue

    cell_id = int(m.group(1))
    t_str   = float(m.group(2))

    found_cells.add(cell_id)

    if t_str not in times:
        times[t_str] = {}
    times[t_str][cell_id] = f
    
CELL_IDS = sorted(found_cells)
all_times=[]

for t_str, cell_map in times.items():

    all_times.append(t_str)         
      
all_times=sorted(all_times, key=int)

################################
#for every timepoint, render a frame
################################

writer = imageio.get_writer(OUT_VIDEO, fps=FPS)

for time in all_times:
    # if time<=80:
    #     continue
    cells = []
    colours = []
    print(time)
    for cell_id, f in times[time].items(): 
        rho = np.loadtxt(f)
        cells.append(rho)
        colours.append(get_cell_c_TS_L(cell_id))

    if not cells:
        continue
    frame = fields_to_rgb_area(cells,colours)
    writer.append_data(frame)
    
writer.close()

0.0
1.0
2.0
3.0
4.0
5.0
6.0
7.0
8.0
9.0
10.0
11.0
12.0
13.0
14.0
15.0
16.0
17.0
18.0
19.0
20.0
21.0
22.0
23.0
24.0
25.0
26.0
27.0
28.0
29.0
30.0
31.0
32.0
33.0
34.0
35.0
36.0
37.0
38.0
39.0
40.0
41.0
42.0
43.0
44.0
45.0
46.0
47.0
48.0
49.0
50.0
51.0
52.0
53.0
54.0
55.0
56.0
57.0
58.0
59.0
60.0
61.0
62.0
63.0
64.0
65.0
66.0
67.0
68.0
69.0
70.0
71.0
72.0
73.0
74.0
75.0
76.0
77.0
78.0
79.0
80.0
81.0
82.0
83.0
84.0
85.0
86.0
87.0
88.0
89.0
90.0
91.0
92.0
93.0
94.0
95.0
96.0
97.0
98.0
99.0
100.0
101.0
102.0


# 0.5 Line visualization

In [35]:
writer = imageio.get_writer(OUT_VIDEO, fps=FPS)

for time in all_times:
    cells = []
    colours = []
    
    for cell_id, f in times[time].items(): 
        rho = np.loadtxt(f)
        cells.append(rho)
        colours.append(get_cell_c(cell_id))

    if not cells:
        continue
    frame = fields_to_rgb_line(cells, colours)

    
    writer.append_data(frame)
writer.close()